# Aeon Ephys Pipeline Guide

End-to-end guide for processing ephys data through the Aeon pipeline.

**Prerequisites:** Complete the setup in [step00_setup.md](step00_setup.md) before running this notebook.

This notebook is a thin wrapper around the guide scripts (`step01_*.py` through `step06_*.py`). Each cell calls one function from a script. The scripts contain all the logic and inline documentation — open them to understand what each step does.

In [ ]:
# Add the guide directory to sys.path so imports work regardless of
# where the notebook is launched from.
import sys
from pathlib import Path

guide_dir = Path("docs/ephys_runbooks").resolve()
if str(guide_dir) not in sys.path:
    sys.path.insert(0, str(guide_dir))

## Configuration

Edit these variables for your experiment. All subsequent cells use these values.

In [ ]:
# ---------- Edit these for your experiment ----------
experiment_name = "abcGolden01-aeonx1"
# abc experiments record behavior and ephys on separate acquisition machines:
#   - AEON3 captures behavior (CameraTop, etc.)  -> registered as "raw"
#   - AEONX1 captures ephys (NeuropixelsV2)       -> registered as "raw-ephys"
raw_behavior_dir = "/ceph/aeon/aeon/data/raw/AEON3/abcGolden01"
raw_ephys_dir = "/ceph/aeon/aeon/data/raw/AEONX1/abcGolden01"
subject = "IAA-1147881"

# Probe info
probe_serial = "23299108854"
probe_type = "neuropixels2.0"
channel_map_file = "M81_ProbeB_4Shanks_1000_to_1700_um.json"

# Block params (test subset)
block_duration_min = 30
overlap_min = 10
n_blocks = 3

# Sorting params
paramset_id = 400
sorting_method = "kilosort4"
sorting_groups = "per_shank"
matching_paramset_id = 1

## Step 1: Register Experiment

Register the experiment, subject, probe type, electrode configuration, and probe assignments. Then ingest epochs and chunks.

In [ ]:
from step01_register_experiment import (
    register_experiment,
    ensure_probe_type,
    create_electrode_config,
    create_probe_assignments,
    register_epochs,
    ingest_chunks,
    verify_registration,
)

register_experiment(experiment_name, raw_behavior_dir, raw_ephys_dir, subject)
ensure_probe_type(probe_type)
create_electrode_config(raw_ephys_dir, channel_map_file, probe_type)
assignments_dir = create_probe_assignments(raw_ephys_dir, probe_serial, subject)
register_epochs(experiment_name, probe_assignments_dir=assignments_dir)
ingest_chunks(experiment_name)
verify_registration(experiment_name)

## Step 2: Define Blocks

Create time windows for spike sorting. Test configuration: 3 blocks of 30 minutes with 10-minute overlap.

In [ ]:
from step02_define_blocks import query_available_data, create_blocks, verify_blocks

query_available_data(experiment_name)
create_blocks(experiment_name, subject, block_duration_min, overlap_min, n_blocks)
verify_blocks(experiment_name)

## Step 3: Spike Sorting

Set up sorting prerequisites, run preprocessing, and generate SLURM submission scripts for GPU sorting.

In [ ]:
from step03_spike_sorting import (
    setup_sorting_prerequisites,
    run_preprocessing,
    write_spike_sorting_scripts,
)

setup_sorting_prerequisites(
    experiment_name, subject, paramset_id, sorting_method,
    sorting_groups=sorting_groups,
    raw_ephys_dir=raw_ephys_dir,
    channel_map_file=channel_map_file,
)
run_preprocessing(experiment_name)
write_spike_sorting_scripts(experiment_name, subject)

### STOP: Submit the SLURM job

The scripts `run_aeon_spike_sorting.py` and `run_aeon_spike_sorting.sh` have been written to the current directory. Submit the sorting job with `sbatch run_aeon_spike_sorting.sh` and wait for it to complete before continuing.

In [ ]:
# Run this cell AFTER the SLURM sorting job has completed.
from step04_post_sorting_and_curation import run_post_sorting, verify_sorting

run_post_sorting(experiment_name)
verify_sorting(experiment_name)

## Step 4: Curation

Auto-approve sorting results (Path A) for testing, or follow the manual curation guide in [step04_post_sorting_and_curation.py](step04_post_sorting_and_curation.py) for production (Path B with SpikeInterface GUI).

The cell below runs auto-approval (Path A). Skip it if you plan to curate manually.

In [ ]:
# Auto-approval (Path A) -- skip if doing manual curation
from step04_post_sorting_and_curation import auto_approve_curation

auto_approve_curation(experiment_name)

## Step 5: Unit Matching

Sync spike times to the behavioral clock and match units across overlapping blocks.

In [ ]:
from step05_unit_matching import sync_spikes, run_unit_matching, verify_matching

sync_spikes(experiment_name)
run_unit_matching(experiment_name, subject, matching_paramset_id)
verify_matching(experiment_name)

## Step 6: Analysis Examples

Fetch spike times and create basic visualizations.

In [ ]:
from step06_analysis_examples import fetch_spike_times, plot_raster, fetch_behavioral_events

spike_dict = fetch_spike_times(experiment_name)
if spike_dict:
    plot_raster(spike_dict)
fetch_behavioral_events(experiment_name)

## Next Steps

- **Analysis:** See analysis repos for deeper analysis workflows
- **Production parameters:** Use 20-hour blocks with 5-hour overlap for real experiments
- **Manual curation:** Follow Path B in [step04_post_sorting_and_curation.py](step04_post_sorting_and_curation.py) for proper unit curation
- **Questions:** Contact the pipeline team